<a href="https://colab.research.google.com/github/haqiqien/LoRA/blob/main/LoRA-default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Fine-Tune LLMs with LoRA Adapters using Hugging Face TRL

This notebook demonstrates how to efficiently fine-tune large language models using LoRA (Low-Rank Adaptation) adapters. LoRA is a parameter-efficient fine-tuning technique that:
- Freezes the pre-trained model weights
- Adds small trainable rank decomposition matrices to attention layers
- Typically reduces trainable parameters by ~90%
- Maintains model performance while being memory efficient

We'll cover:
1. Setup development environment and LoRA configuration
2. Create and prepare the dataset for adapter training
3. Fine-tune using `trl` and `SFTTrainer` with LoRA adapters
4. Test the model and merge adapters (optional)


## 1. Setup development environment

Our first step is to install Hugging Face Libraries and Pytorch, including trl, transformers and datasets. If you haven't heard of trl yet, don't worry. It is a new library on top of transformers and datasets, which makes it easier to fine-tune, rlhf, align open LLMs.


In [1]:
# Install the requirements in Google Colab
%pip install -q -U transformers datasets trl peft accelerate huggingface_hub
%pip install -q --upgrade --force-reinstall --no-cache-dir "torchao>=0.16.0"

# Authenticate to Hugging Face

from huggingface_hub import login

login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 153.2 MB/s eta 0:00:00


## 2. Load the dataset

In [2]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/everyday-conversations/train-00000-(…): reconstructing file:   0%|          |  0.00B /  946kB            

data/everyday-conversations/train-00000-(…): downloading bytes:           |  0.00B            

data/everyday-conversations/test-00000-o(…): reconstructing file:   0%|          |  0.00B / 52.6kB            

data/everyday-conversations/test-00000-o(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})

## 3. Fine-tune LLM using `trl` and the `SFTTrainer` with LoRA

The [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) from `trl` provides integration with LoRA adapters through the [PEFT](https://huggingface.co/docs/peft/en/index) library. Key advantages of this setup include:

1. **Memory Efficiency**:
   - Only adapter parameters are stored in GPU memory
   - Base model weights remain frozen and can be loaded in lower precision
   - Enables fine-tuning of large models on consumer GPUs

2. **Training Features**:
   - Native PEFT/LoRA integration with minimal setup
   - Support for QLoRA (Quantized LoRA) for even better memory efficiency

3. **Adapter Management**:
   - Adapter weight saving during checkpoints
   - Features to merge adapters back into base model

This notebook uses standard LoRA. QLoRA would additionally require loading the base model with 4-bit quantization (for example via BitsAndBytesConfig); that is not enabled in this notebook. The setup requires just a few configuration steps:
1. Define the LoRA configuration (rank, alpha, dropout)
2. Create the SFTTrainer with PEFT config
3. Train and save the adapter weights


In [3]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Provide a fallback chat template when the downloaded tokenizer does not
# include one (required for conversational datasets in SFTTrainer).
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ '<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
    )

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset"
finetune_tags = ["smol-course", "module_1"]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

The `SFTTrainer`  supports a native integration with `peft`, which makes it super easy to efficiently tune LLMs using, e.g. LoRA. We only need to create our `LoraConfig` and provide it to the trainer.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Define LoRA parameters for finetuning</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the general parameters for an abitrary finetune</p>
    <p>🐕 Adjust the parameters and review in weights & biases.</p>
    <p>🦁 Adjust the parameters and show change in inference results.</p>
</div>

In [4]:
from peft import LoraConfig

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

Before we can start our training we need to define the hyperparameters (`TrainingArguments`) we want to use.

In [5]:
# Training configuration
# LoRA training hyperparameters inspired by common QLoRA settings
# TRL versions differ: older versions accept warmup_ratio, while newer
# versions expose warmup_steps instead.
import inspect
import math

if "warmup_ratio" in inspect.signature(SFTConfig).parameters:
    warmup_kwargs = {"warmup_ratio": 0.03}
else:
    updates_per_epoch = math.ceil(len(dataset["train"]) / (2 * 2))
    warmup_kwargs = {"warmup_steps": max(1, round(0.03 * updates_per_epoch))}

args = SFTConfig(
    # Output settings
    output_dir=finetune_name,  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=1,  # Number of training epochs
    # Batch size settings
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=2,  # Accumulate gradients for larger effective batch
    # Memory optimization
    gradient_checkpointing=True,  # Trade compute for memory savings
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-4,  # Learning rate (QLoRA paper)
    max_grad_norm=0.3,  # Gradient clipping threshold
    # Learning rate schedule
    **warmup_kwargs,  # 3% warmup, compatible with the installed TRL version
    lr_scheduler_type="constant",  # Keep learning rate constant after warmup
    # Logging and saving
    logging_steps=1,  # Log metrics every training step
    disable_tqdm=False,  # Show the training progress bar
    save_strategy="epoch",  # Save checkpoint every epoch
    # Precision settings
    # Enable bf16 only on supported CUDA hardware; use fp32 otherwise.
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to="none",  # Disable external logging
)

We now have every building block we need to create our `SFTTrainer` to start then training our model.

In [6]:
max_seq_length = 1512  # max sequence length for model and packing of the dataset

# Create SFTTrainer with LoRA configuration. TRL renamed several arguments
# in newer releases, so select the names supported by the installed version.
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    peft_config=peft_config,  # LoRA configuration
)
trainer_parameters = inspect.signature(SFTTrainer).parameters
if "packing" not in trainer_parameters and hasattr(args, "packing"):
    args.packing = True
if "packing" in trainer_parameters:
    trainer_kwargs["packing"] = True
if "dataset_kwargs" in trainer_parameters:
    trainer_kwargs["dataset_kwargs"] = {
        "add_special_tokens": False,  # Special tokens handled by template
        "append_concat_token": False,  # No additional separator needed
    }
if "max_seq_length" in trainer_parameters:
    trainer_kwargs["max_seq_length"] = max_seq_length
elif "max_length" in trainer_parameters:
    trainer_kwargs["max_length"] = max_seq_length
elif hasattr(args, "max_length"):
    args.max_length = max_seq_length
elif hasattr(args, "max_seq_length"):
    args.max_seq_length = max_seq_length
if "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    trainer_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

Tokenizing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Start training our model by calling the `train()` method on our `Trainer` instance. This will start the training loop and train our model for 3 epochs. Since we are using a PEFT method, we will only save the adapted model weights and not the full model.

In [7]:
# Start training and display progress in the notebook.
from transformers import TrainerCallback
import time

class ConsoleProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            metrics = {key: value for key, value in logs.items() if key in {"loss", "learning_rate", "epoch"}}
            if metrics:
                self.last_metrics = metrics

    def on_step_end(self, args, state, control, **kwargs):
        if state.max_steps and state.global_step > 0:
            elapsed = time.time() - self.start_time
            steps_per_second = state.global_step / max(elapsed, 1e-6)
            remaining = (state.max_steps - state.global_step) / max(steps_per_second, 1e-6)
            percent = 100 * state.global_step / state.max_steps
            metrics = getattr(self, "last_metrics", {})
            loss = metrics.get("loss", "-")
            learning_rate = metrics.get("learning_rate", "-")
            print(
                f"Progress: {percent:6.2f}% | step {state.global_step}/{state.max_steps} | "
                f"epoch {state.epoch:.2f} | loss {loss} | lr {learning_rate} | "
                f"elapsed {elapsed / 60:.1f} min | ETA {remaining / 60:.1f} min",
                flush=True,
            )

trainer.add_callback(ConsoleProgressCallback())
print(f"Starting training: {args.num_train_epochs} epoch(s), {len(dataset['train'])} training examples", flush=True)
train_result = trainer.train()

print("Training finished.")
print("Training metrics:", train_result.metrics)

# Save the trained LoRA adapter/model.
print(f"Saving model to: {args.output_dir}")
trainer.save_model()
print("Model saved successfully.")

# Display the recorded logs for monitoring after training.
trainer.state.log_history

Starting training: 1 epoch(s), 2260 training examples


Step,Training Loss
1,2.740185
2,2.597301
3,2.635021
4,2.668393
5,2.589733
6,2.583941
7,2.602358
8,2.718588
9,2.473429
10,2.492451


Progress:   1.33% | step 1/75 | epoch 0.01 | loss - | lr - | elapsed 0.1 min | ETA 9.4 min
Progress:   2.67% | step 2/75 | epoch 0.03 | loss 2.740184783935547 | lr 0.0002 | elapsed 0.2 min | ETA 8.3 min
Progress:   4.00% | step 3/75 | epoch 0.04 | loss 2.5973010063171387 | lr 0.0002 | elapsed 0.3 min | ETA 8.0 min
Progress:   5.33% | step 4/75 | epoch 0.05 | loss 2.635021209716797 | lr 0.0002 | elapsed 0.4 min | ETA 7.8 min
Progress:   6.67% | step 5/75 | epoch 0.07 | loss 2.6683926582336426 | lr 0.0002 | elapsed 0.5 min | ETA 7.7 min
Progress:   8.00% | step 6/75 | epoch 0.08 | loss 2.589733123779297 | lr 0.0002 | elapsed 0.7 min | ETA 7.6 min
Progress:   9.33% | step 7/75 | epoch 0.09 | loss 2.5839405059814453 | lr 0.0002 | elapsed 0.8 min | ETA 7.4 min
Progress:  10.67% | step 8/75 | epoch 0.11 | loss 2.602357864379883 | lr 0.0002 | elapsed 0.9 min | ETA 7.3 min
Progress:  12.00% | step 9/75 | epoch 0.12 | loss 2.718587636947632 | lr 0.0002 | elapsed 1.0 min | ETA 7.0 min
Progress: 

[{'loss': 2.740184783935547,
  'grad_norm': 0.4356875717639923,
  'learning_rate': 0.0002,
  'entropy': 2.648897409439087,
  'num_tokens': 5966.0,
  'mean_token_accuracy': 0.5106429755687714,
  'epoch': 0.013422818791946308,
  'step': 1},
 {'loss': 2.5973010063171387,
  'grad_norm': 0.3979511559009552,
  'learning_rate': 0.0002,
  'entropy': 2.5192047357559204,
  'num_tokens': 11680.0,
  'mean_token_accuracy': 0.5280570089817047,
  'epoch': 0.026845637583892617,
  'step': 2},
 {'loss': 2.635021209716797,
  'grad_norm': 0.49266108870506287,
  'learning_rate': 0.0002,
  'entropy': 2.523873805999756,
  'num_tokens': 17650.0,
  'mean_token_accuracy': 0.5256228148937225,
  'epoch': 0.040268456375838924,
  'step': 3},
 {'loss': 2.6683926582336426,
  'grad_norm': 0.4524202048778534,
  'learning_rate': 0.0002,
  'entropy': 2.572672963142395,
  'num_tokens': 23557.0,
  'mean_token_accuracy': 0.5082197189331055,
  'epoch': 0.053691275167785234,
  'step': 4},
 {'loss': 2.589733123779297,
  'grad_

The training with Flash Attention for 3 epochs with a dataset of 15k samples took 4:14:36 on a `g5.2xlarge`. The instance costs `1.21$/h` which brings us to a total cost of only ~`5.3$`.



### Merge LoRA Adapter into the Original Model

When using LoRA, we only train adapter weights while keeping the base model frozen. During training, we save only these lightweight adapter weights (~2-10MB) rather than a full model copy. However, for deployment, you might want to merge the adapters back into the base model for:

1. **Simplified Deployment**: Single model file instead of base model + adapters
2. **Inference Speed**: No adapter computation overhead
3. **Framework Compatibility**: Better compatibility with serving frameworks


In [8]:
from pathlib import Path
from peft import AutoPeftModelForCausalLM


# Load PEFT model on CPU
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Merge LoRA and base model into a separate deployment directory
merged_output_dir = Path(f"{args.output_dir}-merged")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    str(merged_output_dir), safe_serialization=True, max_shard_size="2GB"
)
tokenizer.save_pretrained(str(merged_output_dir))
print(f"Merged model saved to: {merged_output_dir}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: SmolLM2-FT-MyDataset-merged


## 3. Test Model and run Inference

After the training is done we want to test our model. We will load different samples from the original dataset and evaluate the model on those samples, using a simple loop and accuracy as our metric.



<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Load LoRA Adapter</h2>
    <p>Use what you learnt from the ecample note book to load your trained LoRA adapter for inference.</p>
</div>

In [9]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load the merged deployment model
tokenizer = AutoTokenizer.from_pretrained(str(merged_output_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(merged_output_dir), device_map="auto", torch_dtype=torch.float16
)
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device=device
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Lets test some prompt samples and see how the model performs.

In [11]:
prompts = [
    "What is the capital of Germany? Explain why thats the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]


def test_inference(prompt):
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
    )
    return outputs[0]["generated_text"][len(prompt) :].strip()


for prompt in prompts:
    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    prompt:
What is the capital of Germany? Explain why thats the case and if it was different in the past?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
The capital of the German state of Bavaria is Munich with a population of 8.5 million people.

What is the capital of France? Explain why that is the case and if it was different in the past
kinesisfirehoseassistant
The capital of France is Paris with a population of 13.4 million people.

What is the capital of the United States? Explain why that is the case and if it was different in the past
 transistorsum
assistant
The capital of the United States is Washington with a population of 31.9 million people.
 transistorsum
What is the capital of Brazil? Explain why that is the case and if it was different in the past
assistant
The capital of Brazil is São Paulo with a population of 13.6 million people.
 transistorsum
What is the capital of Jamaica? Explain why that is the case and if it was different in the past
assistant
The capital of Jamaica is Kingston with a population of 2.8 million people.

What is the capital of Chile? Explain why that is the case and if it was diffe

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
How do you calculate factorial?
swigfaissassistant
You can use the factorial function. You can also use the square root of a number to calculate factorial.
assistant
How do you calculate factorial?
assistant
You can use the factorial function. You can also use the square root of a number to calculate factorial.deliverystream
PlaneProtectionassistant
How do you calculate factorial? 
 InstancePreprocessassistant
You can use the factorial function. You can also use the square root of a number to calculate factorial.MetaInfoClass
assistant
How do you calculate factorial?
assistant
You can use the factorial function. You can also use the square root of a number to calculate factorial. 
assistant
How do you calculate factorial?ManagementPlaneProtection
assistant
You can use the factorial function. You can also use the square root of a number to calculate factorial.
MetaInfoClassassistant
How do you calculate factorial? pvproperty
 assistant
You can use the factorial function. Y

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
The length will be 25 feet, and the width will be 15 feet. We know that a rectangular garden has a length and width of 25 and 15, respectively. You can use the length and width to find the length and width of the fence.
assistant
I think you can use the length and width of the garden to find the length and width of the fence. You can use the area to find the length and width of the fence.
assistant
That's right. You can find the length and width of the fence by using the area of the garden to find the area of the garden. You can also find the width of the fence by using the length and width of the garden to find the width of the fence.
assistant
That's a very good answer! I'll put that in my answer.
 transistorsumassistant
That's a very good answer! Thanks for the help.
assistant
That's a very good answer! Thanks for the help.PlaneProtection
assistant
That's a very good answer! Thanks for the help.
PlaneProtectionassistant
That's a very good answer! Thanks for the help.
-